# 02 — PySide6 : introduction

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- créer une application Qt avec PySide6 ;
- utiliser les widgets (QLabel, QPushButton, QLineEdit, QTextEdit, QCheckBox) ;
- organiser avec les layouts (QVBoxLayout, QHBoxLayout, QGridLayout, QFormLayout) ;
- utiliser les signaux et slots pour la communication entre widgets ;
- créer une MainWindow avec menus, toolbars et statusbar ;
- afficher des boîtes de dialogue (QMessageBox, QFileDialog) ;
- utiliser QTableWidget pour les données tabulaires.

## Prérequis — ce que vous devez maîtriser

À ce stade de la formation, vous savez :

- Python, classes, héritage ;
- notions d’événementiel (callbacks).

Ce que ce notebook **ne couvrira pas** :

- QML et Qt Quick ;
- Qt Designer (générateur d’interfaces) ;
- le déploiement en binaire (PyInstaller, Nuitka).

## Plan

1. Installation
2. Application minimale
3. Widgets de base
4. Layouts
5. Signaux et slots
6. Signaux personnalisés
7. QMainWindow
8. Menus et toolbars
9. Boîtes de dialogue
10. Tables
11. Synthèse
12. Exercices
13. Ressources externes

### Exemple avancé : application avec barre de menu et barre d'état

Une `QMainWindow` permet d'organiser l'application avec une barre de menus,
une barre d'outils, un widget central et une barre d'état.

In [ ]:
from PySide6.QtWidgets import (
    QApplication, QMainWindow, QTextEdit,
    QMenuBar, QStatusBar, QFileDialog, QMessageBox
)
from PySide6.QtGui import QAction
import sys

class NotepadWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle('Mini Notepad')
        self.resize(600, 400)

        # Widget central
        self.editor = QTextEdit()
        self.setCentralWidget(self.editor)

        # Menu
        menu = self.menuBar()
        file_menu = menu.addMenu('&Fichier')

        open_action = QAction('&Ouvrir', self)
        open_action.triggered.connect(self.open_file)
        file_menu.addAction(open_action)

        save_action = QAction('&Sauvegarder', self)
        save_action.triggered.connect(self.save_file)
        file_menu.addAction(save_action)

        # Barre d'état
        self.statusBar().showMessage('Prêt')
        self.editor.textChanged.connect(
            lambda: self.statusBar().showMessage(
                f'{len(self.editor.toPlainText())} caractères'
            )
        )

    def open_file(self):
        path, _ = QFileDialog.getOpenFileName(self, 'Ouvrir')
        if path:
            with open(path) as f:
                self.editor.setPlainText(f.read())

    def save_file(self):
        path, _ = QFileDialog.getSaveFileName(self, 'Sauvegarder')
        if path:
            with open(path, 'w') as f:
                f.write(self.editor.toPlainText())

# app = QApplication(sys.argv)
# win = NotepadWindow()
# win.show()
# app.exec()
print('Mini Notepad prêt (décommentez pour lancer)')

### Exercice : convertisseur de température

Créez une application PySide6 avec :
1. Un `QDoubleSpinBox` pour la température d'entrée
2. Des `QRadioButton` pour choisir la conversion (C->F ou F->C)
3. Un `QLabel` affichant le résultat en temps réel
4. Un `QPushButton` "Réinitialiser"

In [ ]:
from PySide6.QtWidgets import (
    QApplication, QWidget, QVBoxLayout, QHBoxLayout,
    QDoubleSpinBox, QRadioButton, QLabel, QPushButton, QGroupBox
)
import sys

class TemperatureConverter(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle('Convertisseur de température')

        layout = QVBoxLayout()

        # Entrée
        self.spin = QDoubleSpinBox()
        self.spin.setRange(-273.15, 10000)
        self.spin.setValue(0)
        self.spin.setSuffix(' °')
        layout.addWidget(QLabel('Température :'))
        layout.addWidget(self.spin)

        # Direction
        group = QGroupBox('Conversion')
        grp_layout = QHBoxLayout()
        self.c_to_f = QRadioButton('°C -> °F')
        self.f_to_c = QRadioButton('°F -> °C')
        self.c_to_f.setChecked(True)
        grp_layout.addWidget(self.c_to_f)
        grp_layout.addWidget(self.f_to_c)
        group.setLayout(grp_layout)
        layout.addWidget(group)

        # Résultat
        self.result = QLabel('Résultat : 32.00 °F')
        self.result.setStyleSheet('font-size: 16px; font-weight: bold;')
        layout.addWidget(self.result)

        # Bouton reset
        reset_btn = QPushButton('Réinitialiser')
        reset_btn.clicked.connect(lambda: self.spin.setValue(0))
        layout.addWidget(reset_btn)

        self.setLayout(layout)

        # Connexions
        self.spin.valueChanged.connect(self.convert)
        self.c_to_f.toggled.connect(self.convert)

    def convert(self):
        val = self.spin.value()
        if self.c_to_f.isChecked():
            result = val * 9/5 + 32
            self.result.setText(f'Résultat : {result:.2f} °F')
        else:
            result = (val - 32) * 5/9
            self.result.setText(f'Résultat : {result:.2f} °C')

# app = QApplication(sys.argv)
# w = TemperatureConverter()
# w.show()
# app.exec()
print('Convertisseur prêt (décommentez pour lancer)')

### Exercice avancé : liste de tâches (Todo)

Créez une application de gestion de tâches :
1. `QLineEdit` pour saisir une tâche
2. `QListWidget` pour afficher les tâches
3. Les tâches cochées (double-clic) sont barrées
4. Bouton "Supprimer" pour retirer la tâche sélectionnée
5. `QLabel` avec le compteur de tâches restantes

In [ ]:
from PySide6.QtWidgets import (
    QApplication, QWidget, QVBoxLayout, QHBoxLayout,
    QLineEdit, QListWidget, QListWidgetItem, QPushButton, QLabel
)
from PySide6.QtCore import Qt
import sys

class TodoApp(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle('Todo List')
        layout = QVBoxLayout()

        # Saisie
        input_layout = QHBoxLayout()
        self.input_field = QLineEdit()
        self.input_field.setPlaceholderText('Nouvelle tâche...')
        add_btn = QPushButton('Ajouter')
        add_btn.clicked.connect(self.add_task)
        self.input_field.returnPressed.connect(self.add_task)
        input_layout.addWidget(self.input_field)
        input_layout.addWidget(add_btn)
        layout.addLayout(input_layout)

        # Liste
        self.task_list = QListWidget()
        self.task_list.itemDoubleClicked.connect(self.toggle_task)
        layout.addWidget(self.task_list)

        # Actions
        del_btn = QPushButton('Supprimer sélection')
        del_btn.clicked.connect(self.delete_task)
        layout.addWidget(del_btn)

        self.counter = QLabel('0 tâches')
        layout.addWidget(self.counter)

        self.setLayout(layout)

    def add_task(self):
        text = self.input_field.text().strip()
        if text:
            self.task_list.addItem(text)
            self.input_field.clear()
            self.update_count()

    def toggle_task(self, item):
        font = item.font()
        font.setStrikeOut(not font.strikeOut())
        item.setFont(font)
        self.update_count()

    def delete_task(self):
        row = self.task_list.currentRow()
        if row >= 0:
            self.task_list.takeItem(row)
            self.update_count()

    def update_count(self):
        total = self.task_list.count()
        done = sum(1 for i in range(total)
                   if self.task_list.item(i).font().strikeOut())
        self.counter.setText(f'{total - done}/{total} tâches restantes')

# app = QApplication(sys.argv)
# w = TodoApp()
# w.show()
# app.exec()
print('Todo App prête (décommentez pour lancer)')

### Comparaison Tkinter vs PySide6

| Critère | Tkinter | PySide6 |
|---------|---------|--------|
| Installation | Inclus dans Python | `pip install PySide6` |
| Look & feel | Natif (basique) | Natif (professionnel) |
| Signaux/slots | Non (callbacks) | Oui |
| Widgets avancés | Limité | Très riche |
| Designer visuel | Non | Qt Designer |
| Performance | Correcte | Excellente |
| Documentation | Basique | Très complète |
| Licence | PSF | LGPL |

**Conseil** : Tkinter pour les scripts rapides, PySide6 pour les applications professionnelles.

---

## 1. Installation

In [ ]:
# pip install PySide6
import PySide6
print(f"PySide6 {PySide6.__version__}")

---

## 2. Application minimale

`QApplication` doit exister **avant** tout widget.

In [ ]:
# from PySide6.QtWidgets import QApplication, QLabel
# import sys
# app = QApplication(sys.argv)
# label = QLabel('Hello PySide6 !')
# label.show()
# app.exec()
print("QApplication + widget.show() + app.exec()")

### Remarques

- **Un seul** `QApplication` par programme.
- `app.exec()` est bloquant (comme `mainloop()` en Tkinter).
- Dans un notebook, on utilise `%gui qt` ou on montre le code sans l’exécuter.

---

## 3. Widgets de base

In [ ]:
# from PySide6.QtWidgets import (
#     QLabel, QPushButton, QLineEdit, QTextEdit,
#     QCheckBox, QRadioButton, QComboBox, QSpinBox,
#     QSlider, QProgressBar,
# )
print("QLabel, QPushButton, QLineEdit, QTextEdit, QCheckBox...")
print("QComboBox, QSpinBox, QSlider, QProgressBar")

In [ ]:
# QLabel
# label = QLabel('Texte')
# label.setFont(QFont('Arial', 14))
# label.setAlignment(Qt.AlignCenter)
print("QLabel : setFont(), setAlignment()")

In [ ]:
# QPushButton
# btn = QPushButton('Cliquez')
# btn.setEnabled(False)  # grisé
# btn.setToolTip('Info au survol')
print("QPushButton : setEnabled(), setToolTip()")

In [ ]:
# QLineEdit
# line = QLineEdit()
# line.setPlaceholderText('Entrez votre nom...')
# line.setMaxLength(50)
# texte = line.text()
print("QLineEdit : text(), setText(), setPlaceholderText()")

---

## 4. Layouts

In [ ]:
# from PySide6.QtWidgets import QVBoxLayout, QHBoxLayout, QGridLayout, QFormLayout, QWidget
#
# layout = QVBoxLayout()
# layout.addWidget(QLabel('Nom'))
# layout.addWidget(QLineEdit())
# layout.addWidget(QPushButton('OK'))
#
# container = QWidget()
# container.setLayout(layout)
print("QVBoxLayout, QHBoxLayout, QGridLayout, QFormLayout")

| Layout | Description |
|---|---|
| `QVBoxLayout` | Vertical (de haut en bas) |
| `QHBoxLayout` | Horizontal (de gauche à droite) |
| `QGridLayout` | Grille (row, column) |
| `QFormLayout` | Label + champ sur chaque ligne |

In [ ]:
# Imbriquer des layouts
# main_layout = QVBoxLayout()
# top_row = QHBoxLayout()
# top_row.addWidget(QPushButton('A'))
# top_row.addWidget(QPushButton('B'))
# main_layout.addLayout(top_row)
# main_layout.addWidget(QLabel('En dessous'))
print("addLayout() pour imbriquer")

---

## 5. Signaux et slots

C’est le mécanisme central de Qt : un signal émis par un widget est reçu par un slot (callback).

In [ ]:
# btn = QPushButton('Click')
# btn.clicked.connect(lambda: print('Bouton cliqué'))
# line_edit.textChanged.connect(lambda text: print(f'Texte: {text}'))
# checkbox.stateChanged.connect(handler)
print("signal.connect(slot)")

### Signaux courants

| Widget | Signal | Argument |
|---|---|---|
| `QPushButton` | `clicked` | `bool` (checked) |
| `QLineEdit` | `textChanged` | `str` |
| `QLineEdit` | `returnPressed` | *(aucun)* |
| `QCheckBox` | `stateChanged` | `int` |
| `QComboBox` | `currentIndexChanged` | `int` |
| `QSlider` | `valueChanged` | `int` |

---

## 6. Signaux personnalisés

In [ ]:
# from PySide6.QtCore import Signal
#
# class MyWidget(QWidget):
#     value_changed = Signal(int)  # signal custom
#
#     def update_value(self, v: int) -> None:
#         self.value_changed.emit(v)  # émet le signal
print("Signal(type) + emit()")

---

## 7. QMainWindow

Fenêtre complète avec barre de menu, toolbar, statusbar et widget central.

In [ ]:
# from PySide6.QtWidgets import QMainWindow
#
# class Main(QMainWindow):
#     def __init__(self) -> None:
#         super().__init__()
#         self.setWindowTitle('Mon application')
#         self.resize(800, 600)
#         self.setCentralWidget(QLabel('Contenu'))
#         self.statusBar().showMessage('Prêt')
print("QMainWindow + setCentralWidget() + statusBar()")

---

## 8. Menus et toolbars

In [ ]:
# file_menu = self.menuBar().addMenu('Fichier')
# open_action = file_menu.addAction('Ouvrir')
# open_action.setShortcut('Ctrl+O')
# open_action.triggered.connect(self.open_file)
# file_menu.addSeparator()
# file_menu.addAction('Quitter', self.close)
print("menuBar().addMenu().addAction() + setShortcut()")

In [ ]:
# Toolbar
# toolbar = self.addToolBar('Outils')
# toolbar.addAction(open_action)
print("addToolBar() réutilise les mêmes QAction")

---

## 9. Boîtes de dialogue

In [ ]:
# from PySide6.QtWidgets import QMessageBox, QFileDialog
#
# QMessageBox.information(self, 'Info', 'Opération réussie')
# QMessageBox.warning(self, 'Attention', 'Données manquantes')
# reponse = QMessageBox.question(self, 'Confirmer', 'Continuer ?')
print("QMessageBox: information, warning, critical, question")

In [ ]:
# path, _ = QFileDialog.getOpenFileName(
#     self, 'Ouvrir', '', 'Texte (*.txt);;Tous (*.*)',
# )
# path_save, _ = QFileDialog.getSaveFileName(self, 'Enregistrer')
# directory = QFileDialog.getExistingDirectory(self, 'Dossier')
print("QFileDialog: getOpenFileName, getSaveFileName, getExistingDirectory")

---

## 10. Tables

In [ ]:
# from PySide6.QtWidgets import QTableWidget, QTableWidgetItem
#
# table = QTableWidget(3, 2)  # 3 lignes, 2 colonnes
# table.setHorizontalHeaderLabels(['Nom', 'Age'])
# table.setItem(0, 0, QTableWidgetItem('Alice'))
# table.setItem(0, 1, QTableWidgetItem('30'))
print("QTableWidget(rows, cols) + setItem(row, col, QTableWidgetItem)")

In [ ]:
# Lire une cellule
# item = table.item(0, 0)
# if item:
#     print(item.text())
print("table.item(row, col).text()")

---

## Synthèse

| Élément | PySide6 | Notes |
|---|---|---|
| Application | `QApplication(sys.argv)` + `app.exec()` | Un seul par process |
| Fenêtre | `QMainWindow` / `QWidget` | |
| Widgets | `QLabel`, `QPushButton`, `QLineEdit`... | |
| Layouts | `QVBoxLayout`, `QHBoxLayout`, `QGridLayout` | `addWidget()`, `addLayout()` |
| Signaux | `signal.connect(slot)` | Mécanisme central |
| Menus | `menuBar().addMenu().addAction()` | |
| Dialogues | `QMessageBox`, `QFileDialog` | |
| Tables | `QTableWidget` | Ou `QTableView` + modèle |

### Règles à retenir

- **Signal/slot** = communication découplée entre widgets.
- **QMainWindow** pour les apps avec menus/toolbars/statusbar.
- **Layouts** pour disposer les widgets (jamais `setGeometry` à la main).
- **`QApplication`** doit exister avant tout widget.
- **PySide6 = binding officiel Qt** (license LGPL).

---

---

## QListWidget

In [ ]:
# from PySide6.QtWidgets import QListWidget
# list_widget = QListWidget()
# list_widget.addItems(['Alpha', 'Beta', 'Gamma'])
# list_widget.currentItemChanged.connect(
#     lambda current, prev: print(f'Selection: {current.text()}')
# )
print('QListWidget: addItems(), currentItemChanged signal')

In [ ]:
# Ajouter/supprimer dynamiquement
# list_widget.addItem('Delta')
# list_widget.takeItem(0)  # supprime le premier
# count = list_widget.count()
print('addItem(), takeItem(), count()')

---

## QTimer

In [ ]:
# from PySide6.QtCore import QTimer
# timer = QTimer()
# timer.setInterval(1000)  # ms
# timer.timeout.connect(lambda: print('tick'))
# timer.start()
# timer.stop()
print('QTimer: setInterval(), start(), stop(), timeout signal')

Utile pour les mises a jour periodiques (horloge, animation, polling).

---

## QStatusBar et QToolBar

In [ ]:
# self.statusBar().showMessage('Pret', timeout=3000)  # 3 secondes
# toolbar = self.addToolBar('Outils')
# toolbar.addAction('Nouveau', self.new_file)
# toolbar.addSeparator()
# toolbar.setMovable(False)
print('statusBar().showMessage(), addToolBar()')

---

## QTabWidget

In [ ]:
# from PySide6.QtWidgets import QTabWidget
# tabs = QTabWidget()
# tabs.addTab(QLabel('Contenu 1'), 'Onglet 1')
# tabs.addTab(QLabel('Contenu 2'), 'Onglet 2')
# tabs.currentChanged.connect(lambda i: print(f'Onglet {i}'))
print('QTabWidget: addTab(), currentChanged signal')

---

## Raccourcis clavier (QShortcut)

In [ ]:
# from PySide6.QtGui import QShortcut, QKeySequence
# shortcut = QShortcut(QKeySequence('Ctrl+N'), self)
# shortcut.activated.connect(self.new_file)
print('QShortcut(QKeySequence(...), parent) + activated.connect()')

---

## Styles et palettes

In [ ]:
# from PySide6.QtWidgets import QStyleFactory
# print(QStyleFactory.keys())  # styles disponibles
# app.setStyle('Fusion')
print('QStyleFactory.keys() pour lister les styles disponibles')

## Exercices

### Exercice 1 *(facile)* — Fenêtre Hello Qt

Créez une fenêtre avec un `QLabel` affichant « Bonjour Qt ».

In [ ]:
from PySide6.QtWidgets import QApplication, QLabel

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from PySide6.QtWidgets import QApplication, QLabel
import sys

app = QApplication(sys.argv)
label = QLabel('Bonjour Qt')
label.show()
# app.exec()
```

</details>

### Exercice 2 *(facile)* — Bouton + handler

Fenêtre avec un bouton qui affiche « Cliqué ! » dans la console.

In [ ]:
# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from PySide6.QtWidgets import QApplication, QPushButton
import sys

app = QApplication(sys.argv)
btn = QPushButton('Cliquez')
btn.clicked.connect(lambda: print('Cliqué !'))
btn.show()
# app.exec()
```

</details>

### Exercice 3 *(moyen)* — Formulaire avec QFormLayout

Formulaire (Nom, Age, Email) avec un bouton Envoyer qui affiche les valeurs.

In [ ]:
# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from PySide6.QtWidgets import (QApplication, QWidget, QFormLayout,
                               QLineEdit, QPushButton)
import sys

app = QApplication(sys.argv)
w = QWidget()
layout = QFormLayout()
nom = QLineEdit()
age = QLineEdit()
email = QLineEdit()
layout.addRow('Nom:', nom)
layout.addRow('Age:', age)
layout.addRow('Email:', email)
btn = QPushButton('Envoyer')
btn.clicked.connect(lambda: print(nom.text(), age.text(), email.text()))
layout.addWidget(btn)
w.setLayout(layout)
w.show()
# app.exec()
```

</details>

### Exercice 4 *(moyen)* — QMainWindow avec menu

MainWindow avec menu Fichier → Quitter (raccourci Ctrl+Q) qui ferme l’appli.

In [ ]:
# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from PySide6.QtWidgets import QApplication, QMainWindow, QLabel
import sys

app = QApplication(sys.argv)
w = QMainWindow()
w.setWindowTitle('Demo')
w.setCentralWidget(QLabel('Contenu'))
file_menu = w.menuBar().addMenu('Fichier')
quit_action = file_menu.addAction('Quitter')
quit_action.setShortcut('Ctrl+Q')
quit_action.triggered.connect(w.close)
w.show()
# app.exec()
```

</details>

### Exercice 5 *(difficile)* — Table dynamique

Table 3 colonnes (Nom, Age, Email). Formulaire en haut, bouton Ajouter qui insère une ligne.

In [ ]:
# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from PySide6.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout,
    QTableWidget, QTableWidgetItem, QLineEdit, QPushButton,
)
import sys

app = QApplication(sys.argv)
w = QMainWindow()
central = QWidget()
layout = QVBoxLayout()
table = QTableWidget(0, 3)
table.setHorizontalHeaderLabels(['Nom', 'Age', 'Email'])
nom = QLineEdit()
nom.setPlaceholderText('Nom')
age = QLineEdit()
age.setPlaceholderText('Age')
email = QLineEdit()
email.setPlaceholderText('Email')
btn = QPushButton('Ajouter')

def add_row():
    r = table.rowCount()
    table.insertRow(r)
    table.setItem(r, 0, QTableWidgetItem(nom.text()))
    table.setItem(r, 1, QTableWidgetItem(age.text()))
    table.setItem(r, 2, QTableWidgetItem(email.text()))
    nom.clear()
    age.clear()
    email.clear()

btn.clicked.connect(add_row)
for widget in [table, nom, age, email, btn]:
    layout.addWidget(widget)
central.setLayout(layout)
w.setCentralWidget(central)
w.resize(500, 400)
w.show()
# app.exec()
```

</details>

### Approfondir : QTableWidget pour afficher des données

`QTableWidget` permet d'afficher des données tabulaires.
C'est l'équivalent Qt d'un tableur basique.

In [ ]:
from PySide6.QtWidgets import (
    QApplication, QWidget, QVBoxLayout,
    QTableWidget, QTableWidgetItem, QHeaderView
)
import sys

class DataTable(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle('Données')
        layout = QVBoxLayout()

        # Données
        data = [
            ('Alice', 28, 'Paris', 45000),
            ('Bob', 35, 'Lyon', 52000),
            ('Charlie', 42, 'Marseille', 48000),
            ('Diana', 31, 'Toulouse', 51000),
        ]
        headers = ['Nom', 'Âge', 'Ville', 'Salaire']

        table = QTableWidget(len(data), len(headers))
        table.setHorizontalHeaderLabels(headers)
        table.horizontalHeader().setSectionResizeMode(QHeaderView.Stretch)

        for row, record in enumerate(data):
            for col, value in enumerate(record):
                table.setItem(row, col, QTableWidgetItem(str(value)))

        layout.addWidget(table)
        self.setLayout(layout)

# app = QApplication(sys.argv)
# w = DataTable()
# w.show()
# app.exec()
print('DataTable prête')

### Exercice : QTimer pour animation

1. Créez un widget qui affiche un compteur s'incrémentant chaque seconde
2. Ajoutez une barre de progression (`QProgressBar`) qui se remplit en 30 secondes
3. Boutons Pause/Reprendre

In [ ]:
from PySide6.QtWidgets import (
    QApplication, QWidget, QVBoxLayout,
    QLabel, QPushButton, QProgressBar
)
from PySide6.QtCore import QTimer
import sys

class TimerWidget(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle('Timer')
        layout = QVBoxLayout()

        self.counter = 0
        self.label = QLabel('0 secondes')
        self.label.setStyleSheet('font-size: 24px;')
        layout.addWidget(self.label)

        self.progress = QProgressBar()
        self.progress.setRange(0, 30)
        layout.addWidget(self.progress)

        self.timer = QTimer()
        self.timer.setInterval(1000)
        self.timer.timeout.connect(self.tick)

        self.btn = QPushButton('Démarrer')
        self.btn.clicked.connect(self.toggle)
        layout.addWidget(self.btn)

        self.setLayout(layout)

    def tick(self):
        self.counter += 1
        self.label.setText(f'{self.counter} secondes')
        self.progress.setValue(min(self.counter, 30))

    def toggle(self):
        if self.timer.isActive():
            self.timer.stop()
            self.btn.setText('Reprendre')
        else:
            self.timer.start()
            self.btn.setText('Pause')

# app = QApplication(sys.argv)
# w = TimerWidget()
# w.show()
# app.exec()
print('TimerWidget prêt')

### Exercice avancé : signaux personnalisés entre widgets

1. Créez un `QSlider` horizontal (0-100)
2. Connectez-le à un `QLabel` qui affiche la valeur
3. Connectez-le aussi à un `QProgressBar`
4. Ajoutez un signal personnalisé qui émet quand la valeur dépasse 80

In [ ]:
from PySide6.QtWidgets import (
    QApplication, QWidget, QVBoxLayout,
    QSlider, QLabel, QProgressBar
)
from PySide6.QtCore import Qt, Signal
import sys

class SliderDemo(QWidget):
    threshold_reached = Signal(int)

    def __init__(self):
        super().__init__()
        self.setWindowTitle('Slider Demo')
        layout = QVBoxLayout()

        self.slider = QSlider(Qt.Horizontal)
        self.slider.setRange(0, 100)
        layout.addWidget(self.slider)

        self.label = QLabel('Valeur : 0')
        layout.addWidget(self.label)

        self.progress = QProgressBar()
        self.progress.setRange(0, 100)
        layout.addWidget(self.progress)

        self.alert = QLabel('')
        self.alert.setStyleSheet('color: red; font-weight: bold;')
        layout.addWidget(self.alert)

        # Connexions
        self.slider.valueChanged.connect(self.on_value_changed)
        self.threshold_reached.connect(
            lambda v: self.alert.setText(f'Seuil atteint ! ({v})')
        )

        self.setLayout(layout)
        self._above = False

    def on_value_changed(self, value):
        self.label.setText(f'Valeur : {value}')
        self.progress.setValue(value)
        if value > 80 and not self._above:
            self._above = True
            self.threshold_reached.emit(value)
        elif value <= 80:
            self._above = False
            self.alert.setText('')

# app = QApplication(sys.argv)
# w = SliderDemo()
# w.show()
# app.exec()
print('SliderDemo prêt')

### Exercice bonus : style avec QSS (Qt Style Sheets)

Qt supporte un langage de style similaire au CSS.
Personnalisez l'apparence d'un widget avec QSS.

In [ ]:
from PySide6.QtWidgets import QApplication, QWidget, QPushButton, QVBoxLayout
import sys

class StyledWidget(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle('QSS Demo')
        layout = QVBoxLayout()

        for label, color in [('Primary', '#2196F3'), ('Success', '#4CAF50'),
                              ('Warning', '#FF9800'), ('Danger', '#F44336')]:
            btn = QPushButton(label)
            btn.setStyleSheet(f'''
                QPushButton {{
                    background-color: {color};
                    color: white;
                    border: none;
                    padding: 10px 20px;
                    border-radius: 5px;
                    font-size: 14px;
                }}
                QPushButton:hover {{
                    background-color: {color}CC;
                }}
                QPushButton:pressed {{
                    background-color: {color}99;
                }}
            ''')
            layout.addWidget(btn)

        self.setLayout(layout)

# app = QApplication(sys.argv)
# w = StyledWidget()
# w.show()
# app.exec()
print('QSS demo prête')

### Pour aller plus loin avec PySide6

- **Qt Designer** : éditeur visuel d'interfaces (inclus dans PySide6)
- **QML** : langage déclaratif pour UIs modernes et animées
- **Model/View** : `QAbstractTableModel` pour les grandes quantités de données
- **QThread** : exécution parallèle pour ne pas bloquer l'UI
- **QChart** : graphiques intégrés (pas besoin de matplotlib)

---

## Ressources externes

- [PySide6 — documentation officielle](https://doc.qt.io/qtforpython-6/)
- [Qt for Python tutorials](https://doc.qt.io/qtforpython-6/tutorials/index.html)
- [Qt Widgets gallery](https://doc.qt.io/qt-6/gallery.html)